# IMERG Data

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

In [ ]:
from pansat.time import TimeRange
from pansat.utils import resample_data_binned

In [ ]:
from pansat.utils import make_latlon_area
lon_min = -125
lat_min = 24
lon_max = -65
lat_max = 51
imerg_grid = make_latlon_area(
    lon_min,
    lat_min,
    lon_max,
    lat_max,
    (lon_max - lon_min) / 0.1, 
    (lat_max - lat_min) / 0.1
)
imerg_grid

In [ ]:
from datetime import datetime
from gprof_ir.imerg import load_imerg_data
from pansat.products.satellite import gpm
from pansat.utils import resample_data_binned

def extract_imerg_data(year, month, day, hour) -> xr.Dataset:
    """
    Download IMERG data and map to CONUS domain.

    Args:
        year: The year
        month: the month
        day: the day
        hout the hour

    Return:
        The IMERG multi sensor precip rate and the radar quality index half an hour after the full hour.
    """
    imerg_data = []
    for minute in [15, 45]:
        time = datetime(year, month, day, hour, minute=minute)
        rec = gpm.l3b_hhr_3imerg_ms_mrg_07b.get(time)[0]
        print(rec)
        data = load_imerg_data(rec.local_path, bounds=(lon_min, lon_max, lon_max, lat_max))
        data = resample_data_binned(data, imerg_grid)
        imerg_data.append(data)
    imerg_data = xr.concat(imerg_data, dim="time")
    return imerg_data.mean("time")
    

In [ ]:
output_path = Path("/gdata2/simon/gprof_ir/conus/imerg")
output_path.mkdir(parents=True, exist_ok=True)

In [ ]:
start_time = np.datetime64("2022-01-01")
end_time = np.datetime64("2022-07-01")

IMERG_VARS = ["surface_precip", "surface_precip_ir", "surface_precip_mw"]

for hour in np.arange(start_time, end_time, np.timedelta64(1, "h")):
    date = hour.astype("datetime64[s]").item()
    print(date)
    output_file = output_path / date.strftime("imerg_%Y%m%d%H%M%S.nc")
    if not output_file.exists():
        try:
            imerg_data = extract_imerg_data(date.year, date.month, date.day, date.hour)[IMERG_VARS]
            for var in IMERG_VARS:
                imerg_data[var].encoding = {
                    "zlib": True,
                    "dtype": "float32"
                }
            imerg_data.to_netcdf(output_file)
        except Exception:
            print("Error processing date: ", date)